Inspeciona O QUE o agente aprendeu: Q-table, visitas e uma trajetória gulosa

In [1]:
import random
from collections import Counter
import numpy as np
from immune_rl.graph import build_pilot_graph
from immune_rl.dynamics import SimpleDynamics, PathogenInflammationReward
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import QLearningPolicy
from immune_rl.encoding import StateEncoder
from immune_rl.entities import Action

In [2]:
SEED = 0
TRAIN_EPISODES = 1500
EVAL_EPISODES = 200
LEVELS = ["baixo", "médio", "alto"]

In [5]:
def make_env():
    return ImmuneEnvironment(build_pilot_graph, SimpleDynamics(), PathogenInflammationReward())
 
 
def train(env, policy):
    for _ in range(TRAIN_EPISODES):
        obs = env.reset()
        while True:
            actions = {cid: policy.select_action(o) for cid, o in obs.items()}
            next_obs, rewards, terminated, truncated, _ = env.step(actions)
            for cid in obs:
                policy.update(obs[cid], actions[cid], rewards[cid], next_obs[cid], terminated)
            obs = next_obs
            if terminated or truncated:
                break
        policy.end_episode()
 
 
def describe(state, encoder):
    names = ["ativação"] + [t.value for t in encoder.neighbor_edges]
    return "  ".join(f"{n}={LEVELS[v] if v < len(LEVELS) else v}" for n, v in zip(names, state))
 
 
def count_visits(env, policy):
    """Roda episódios gulosos e conta quantas vezes cada estado foi visto."""
    visits, chosen = Counter(), Counter()
    for _ in range(EVAL_EPISODES):
        obs = env.reset()
        while True:
            actions = {}
            for cid, o in obs.items():
                visits[policy.encoder.encode(o)] += 1
                actions[cid] = policy.select_action(o)
                chosen[actions[cid]] += 1
            obs, _, terminated, truncated, _ = env.step(actions)
            if terminated or truncated:
                break
    return visits, chosen
 
 
def print_q_table(policy, visits):
    print(f"\n{'estado':<48}{'visitas':>8}  {'NOOP':>7}{'ATIVAR':>8}{'REGULAR':>9}   melhor")
    for state in sorted(policy.q, key=lambda s: -visits[s]):
        q = policy.q[state]
        best = "—  (sem dados)" if np.allclose(q, 0) else Action(int(np.argmax(q))).name
        note = "" if visits[state] else "  (só visto na exploração)"
        print(f"{describe(state, policy.encoder):<48}{visits[state]:>8}  "
              f"{q[0]:>7.2f}{q[1]:>8.2f}{q[2]:>9.2f}   {best}{note}")
 
 
def print_trajectory(env, policy):
    print("\nUm episódio guloso (ε=0):")
    obs = env.reset()
    while True:
        actions = {cid: policy.select_action(o) for cid, o in obs.items()}
        obs, _, terminated, truncated, info = env.step(actions)
        acts = ", ".join(a.name for a in actions.values())
        act = np.mean([c.activation for c in env.graph.cells()])

        if terminated or truncated:
            print("  fim:", "patógeno eliminado" if terminated else "tempo esgotado")
            break

In [6]:
random.seed(SEED)
env = make_env()
policy = QLearningPolicy(StateEncoder(), seed=SEED)
train(env, policy)
policy.epsilon = 0.0

visits, chosen = count_visits(env, policy)
total = sum(chosen.values())

print("Ações escolhidas na avaliação:",
          {a.name: f"{chosen[a] / total:.0%}" for a in Action})

print_q_table(policy, visits)
print_trajectory(env, policy)

Ações escolhidas na avaliação: {'NOOP': '0%', 'ACTIVATE': '92%', 'REGULATE': '8%'}

estado                                           visitas     NOOP  ATIVAR  REGULAR   melhor
ativação=alto  pathogen=baixo  cytokine=alto        2400     0.27    0.33     0.32   ACTIVATE
ativação=baixo  pathogen=alto  cytokine=baixo        800     0.62    0.65     0.61   ACTIVATE
ativação=alto  pathogen=médio  cytokine=alto         800     0.42    0.49     0.45   ACTIVATE
ativação=médio  pathogen=alto  cytokine=baixo        400     0.62    0.62     0.64   REGULATE
ativação=médio  pathogen=alto  cytokine=médio        400     0.58    0.62     0.57   ACTIVATE
ativação=médio  pathogen=médio  cytokine=médio       400     0.54    0.58     0.54   ACTIVATE
ativação=baixo  pathogen=alto  cytokine=médio          0     0.57    0.61     0.57   ACTIVATE  (só visto na exploração)
ativação=baixo  pathogen=médio  cytokine=médio         0     0.49    0.56     0.51   ACTIVATE  (só visto na exploração)
ativação=baixo  path